[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/particle_charges/particle_charges.ipynb)

# Spin(10) Particle Charges

In a Spin(10) model, one generation of quarks and leptons, sixteen particles with all their charges, are the components of a single spinor of ten internal directions. Colour and weak rotations connect members of that family, and the remaining rotations of Spin(10) connect quarks with leptons.

The spinor lives in a left ideal of the algebra: left multiplication by an even element keeps it there, so every internal rotation, `action(element)`, maps states to states, and so does every charge built from those rotations. A particle sector is a part of the spinor on which the commuting charges have definite values, and how strongly a family of rotations connects two sectors is read off the maps between them.

The charge table recovers the Standard Model's charges, and the diagrams show which sectors colour, weak and unifying rotations join.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from functools import reduce
from itertools import product
from operator import mul

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.util.animation import save_figure
from examples.quantum.particle_charges import render

In [ ]:
ga = Algebra("a+b+c+d+e+f+g+h+i+j+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Bivector = ga.gatype.bivector()
Even = ga.gatype.even()
Full = ga.gatype.full()
# One independent blade from each group of blades the ideal identifies.
State = Full.derive.within("b c d f j")
StateMap = ga.gatype((State, State))

## 1. The spinor space

The spinor space must be kept by internal rotations and carry a common phase. A left ideal is such a space: left multiplication keeps every state within it. Five commuting projectors, each one plus an element squaring to one, cut the algebra down to an ideal of sixteen amplitudes with two real phase directions each, and the volume element, which squares to minus one and commutes with every rotation, turns one phase direction into the other.

In [ ]:
# Each factor is a projector; their product selects the spinor representation.
ideal = (1 + mv.a) * (1 + mv.bcde) * (1 + mv.bcfg) * (1 + mv.bdfh) * (1 + mv.cdfi) / 32
embedding = State * ideal                                        # [] Full <- State
# The chart's coefficients each appear with weight 1/32 in the ideal.
readout = (32 * Full).cast(State)                                # [] State <- Full
phase = mv.abcdefghij                                            # [] Even


def action(element: Even) -> StateMap:
    # Multiplication keeps the ideal; the readout returns its independent coefficients.
    return readout(element * embedding)                          # [...] State <- State

## 2. Charge sectors

Each of five planes, turned by the phase, is a sign that splits the spinor in two, and its occupation, `(State - action(signs)) / 2`, counts which half a state lies in. The first three planes carry colour and the last two weak isospin; hypercharge is a weighted sum of all five occupations. Fixing four signs picks one of sixteen sectors, and the fifth follows from the spinor's chirality.

In [ ]:
# The sixteen sign patterns have even occupation parity, selecting one chirality.
occupations = np.array([pattern + (sum(pattern) % 2,) for pattern in product((0, 1), repeat=4)])
# Hypercharge uses the convention in which electric charge is weak isospin plus hypercharge.
hypercharge_weights = np.array([-1 / 3, -1 / 3, -1 / 3, 1 / 2, 1 / 2])

In [ ]:
planes = stack((mv.ab, mv.cd, mv.ef, mv.gh, mv.ij))              # [planes] Bivector
signs = -phase * planes                                          # [planes] Even
occupation = (State - action(signs)) / 2                         # [planes] State <- State

# Each sign projector halves the spinor; four leave two real phase directions.
factors = (1 + signs[None, :4] * (1 - 2 * occupations[:, :4])) / 2   # [sectors, signs] Even
selectors = reduce(mul, (factors[:, i] for i in range(4)))       # [sectors] Even
projectors = action(selectors)                                   # [sectors] State <- State

hypercharge = (hypercharge_weights * occupation).sum(axis=0)     # [] State <- State
weak_isospin = (occupation[3] - occupation[4]) / 2               # [] State <- State
right_isospin = (occupation[3] + occupation[4] - State) / 2      # [] State <- State
electric_charge = hypercharge + weak_isospin                     # [] State <- State
# Two commuting colour readings separate the three colours within each quark multiplet.
colour = stack((
    (occupation[0] - occupation[1]) / 2,
    (occupation[0] + occupation[1] - 2 * occupation[2]) / (2 * np.sqrt(3)),
))                                                               # [colour weights] State <- State


def sector_value(observable: StateMap, sectors: StateMap) -> Scalar:
    # A commuting observable has one value on the sector; its trace there over the sector's rank.
    return observable(sectors).trace() / sectors.trace()         # [...] Scalar


charges = sector_value(electric_charge, projectors)              # [sectors] Scalar
hypercharges = sector_value(hypercharge, projectors)             # [sectors] Scalar
left_weights = sector_value(weak_isospin, projectors)            # [sectors] Scalar
right_weights = sector_value(right_isospin, projectors)          # [sectors] Scalar
colour_weights = sector_value(colour[None, :], projectors[:, None])   # [sectors, colour weights] Scalar

In [ ]:
print("charges:", charges.to_array())
print("hypercharges:", hypercharges.to_array())
print("left isospins:", left_weights.to_array())

The sixteen sectors group into eight kinds of particle. Labels ending in `^c` are the charge conjugates of right-handed fields, with charges opposite to those of the particles, and the neutral singlet is the right-handed neutrino this model includes. [Baez and Huerta](https://arxiv.org/html/0904.1556#S3.SS5) describe this assignment.

In Lie-group representation notation, the states form the chiral $\mathbf{16}$ of $\operatorname{Spin}(10)$, and electric charge is normalized as $Q=T_3+Y$.

## 3. Colour and weak actions

Colour rotations mix the three colours of each quark, and weak rotations mix the two members of each left-handed doublet; leptons are untouched by colour, and the conjugate singlets by the weak rotations. The strength with which a family of rotations connects two sectors is the size of the maps it carries from one into the other.

In [ ]:
# Paired turns exchange two occupations while keeping their sum.
colour_generators = stack((
    mv.ac + mv.bd, mv.ad - mv.bc,
    mv.ae + mv.bf, mv.af - mv.be,
    mv.ce + mv.df, mv.cf - mv.de,
    mv.ab - mv.cd,
    (mv.ab + mv.cd - 2 * mv.ef) / np.sqrt(3),
)) / 4                                                           # [colour generators] Bivector
weak_generators = stack((mv.gi + mv.hj, mv.gj - mv.hi, mv.gh - mv.ij)) / 4   # [weak generators] Bivector
colour_actions = action(colour_generators)                       # [colour generators] State <- State
weak_actions = action(weak_generators)                           # [weak generators] State <- State


def transition_strength(generators: StateMap, sectors: StateMap) -> Scalar:
    # Each block takes a source sector through a generator into a target sector.
    blocks = sectors[:, None, None](generators[None, None, :](sectors[None, :, None]))
    # The squared norm sums over the input directions; two real directions make one amplitude.
    return blocks.scalar_norm_squared().contract().sum(axis=-1) / 2   # [target, source] Scalar


colour_links = transition_strength(colour_actions, projectors)   # [target, source] Scalar
weak_links = transition_strength(weak_actions, projectors)       # [target, source] Scalar
figure = render.draw_connections(occupations, left_weights, right_weights, colour_weights,
                                 stack((colour_links, weak_links)))
save_figure(figure, "particle_charges_gauge")
figure

The left panel shows the colour connections and the right panel the weak ones. Red, green and blue nodes are quarks and gold nodes leptons; filled nodes form the left-handed doublets in the horizontal groups, and hollow nodes are the conjugate singlets in the vertical groups. Positions come from the weak and colour readings, and edges mark nonzero actions, not reaction rates.

## 4. Connections between multiplets

The other rotations of Spin(10) connect groups that colour and weak rotations keep apart. A rotation mixing a colour direction with a weak one joins quark and lepton sectors within the same spinor.

In [ ]:
mixing_generator = (mv.ag + mv.bh) / 4                           # [] Bivector
mixing_action = action(mixing_generator)                         # [] State <- State
mixing_links = transition_strength(mixing_action[None], projectors)   # [target, source] Scalar
figure = render.draw_connections(occupations, left_weights, right_weights, colour_weights, mixing_links[None])
save_figure(figure, "particle_charges_unified")
figure

The new connections cross the groups that colour and weak rotations preserve: edges between gold and coloured nodes join leptons with quarks, and the others join quark sectors with different weak assignments. That they act as physical interactions is a further assumption of a unified model.

In [ ]:
# checks
np.testing.assert_allclose((charges - hypercharges - left_weights).to_array(), 0, atol=1e-12)
connected = [np.count_nonzero(np.triu(links.to_array(), 1) > 1e-12)
             for links in (colour_links, weak_links, mixing_links)]
assert connected == [12, 4, 4]